In [1]:
import unicodedata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------
# 1. 前処理関数（全角半角の統一やスペース除去で精度底上げ）
# ---------------------------------------------------------
def clean_text(text: str) -> str:
    # 全角英数・カナを半角/全角に正規化 (NFKC)
    text = unicodedata.normalize("NFKC", text)
    # 余分な空白を除去
    text = "".join(text.split())
    # 小文字化
    return text.lower()


# ---------------------------------------------------------
# 2. モデルのロード
# ---------------------------------------------------------
# multilingual-e5 は検索・類似度判定で非常に高精度なモデルです
MODEL_NAME = "intfloat/multilingual-e5-base"
print(f"モデル '{MODEL_NAME}' を読み込んでいます...")
model = SentenceTransformer(MODEL_NAME)


# ---------------------------------------------------------
# 3. 商品名の一致判定クラス
# ---------------------------------------------------------
class ProductMatcher:
    def __init__(self, model: SentenceTransformer, threshold: float = 0.82):
        self.model = model
        self.threshold = threshold  # 同一商品とみなすスコアの境界値

    def compare(self, name1: str, name2: str) -> dict:
        # 前処理
        clean_name1 = clean_text(name1)
        clean_name2 = clean_text(name2)

        # E5モデル特有のルール: 比較時はテキストの頭に 'query: ' を追加する
        # （※ Sup-SimCSE や他のモデルを使う場合は 'query: ' は不要です）
        input1 = f"query: {clean_name1}"
        input2 = f"query: {clean_name2}"

        # ベクトル化（エンコード）
        embeddings = self.model.encode([input1, input2], normalize_embeddings=True)

        # コサイン類似度の計算（normalize_embeddings=True の場合は内積で計算可能）
        similarity_score = float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

        # 判定結果の整理
        is_match = similarity_score >= self.threshold

        return {
            "name1": name1,
            "name2": name2,
            "score": round(similarity_score, 4),
            "is_match": is_match
        }


# ---------------------------------------------------------
# 4. 実際のテスト実行
# ---------------------------------------------------------
if __name__ == "__main__":
    matcher = ProductMatcher(model=model, threshold=0.82)

    # 検証したい商品名のペアリスト
    test_pairs = [
        ("ベビースターラーメンチキン味", "ベビースターチキン"),
        ("ベビースターラーメン チキン味 68g", "ベビースターチキン68g"),
        ("ポテトチップス うすしお味", "ポテトチップスコンソメ味"),
        ("コカ・コーラ 500ml PET", "コカコーラ 500ml"),
        ("アサヒ スーパードライ 350ml", "サッポロ 黒ラベル 350ml")
    ]

    print("\n--- 判定結果 ---")
    for p1, p2 in test_pairs:
        result = matcher.compare(p1, p2)
        match_status = "✅ 同一商品" if result["is_match"] else "❌ 判別（別商品）"
        print(f"[{match_status}] スコア: {result['score']:.4f}")
        print(f"   商品A: {result['name1']}")
        print(f"   商品B: {result['name2']}\n")

ModuleNotFoundError: No module named 'sentence_transformers'